# Ejercicio 1

## Cálculo del corrimiento al rojo de NGC5406

Utilizando el espectro obtenido para la galaxia **NGC5406**, identifique al menos **tres líneas espectrales** visibles.

Con esta información, determine el **corrimiento al rojo (redshift)** de la galaxia mediante la relación

$$
z=\frac{\lambda_{\mathrm{obs}}-\lambda_{0}}{\lambda_{0}},
$$

donde:

- $\lambda_{\mathrm{obs}}$ es la longitud de onda observada.
- $\lambda_{0}$ es la longitud de onda en reposo de la línea espectral.

Finalmente, calcule el valor promedio del corrimiento al rojo utilizando las tres líneas identificadas.

In [ ]:
#Ejercicio 1

import numpy as np
import matplotlib.pyplot as plt

#Longitudes de onda y flujo del espectro

wavelength = 10**spectra_data["loglam"]
flux = spectra_data["flux"]

#Líneas espectrales a utilizar

selected_lines = ["[O_II] 3727", "[O_III] 5007", "H_alpha"]

print("Líneas identificadas\n")

redshifts = []

for name in selected_lines:

    mask = lines["LINENAME"] == name

    lambda_obs = lines["LINEWAVE"][mask][0]

    if name == "[O_II] 3727":
        lambda_rest = 3727.0
    elif name == "[O_III] 5007":
        lambda_rest = 5007.0
    elif name == "H_alpha":
        lambda_rest = 6562.8

    z = (lambda_obs - lambda_rest) / lambda_rest

    redshifts.append(z)

    print(f"{name}")
    print(f"  λ_rest = {lambda_rest:.1f} Å")
    print(f"  λ_obs  = {lambda_obs:.2f} Å")
    print(f"  z      = {z:.6f}\n")

#Redshift promedio

z_mean = np.mean(redshifts)

print("-"*40)
print(f"Redshift promedio = {z_mean:.6f}")

#Gráfica del espectro

plt.figure(figsize=(10,5))
plt.plot(wavelength, flux, color="black")

colors = ["royalblue", "crimson", "green"]

for name, color in zip(selected_lines, colors):

    mask = lines["LINENAME"] == name
    lambda_obs = lines["LINEWAVE"][mask][0]

    plt.axvline(
        lambda_obs,
        color=color,
        linestyle="--",
        label=f"{name} ({lambda_obs:.1f} Å)"
    )

plt.xlabel("Longitud de onda (Å)")
plt.ylabel("Flujo")
plt.title(f"Espectro de {galaxy_name}")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

# Ejercicio 2

## Visualización de NGC5406 en los filtros fotométricos del SDSS

Descargue las imágenes FITS correspondientes a los filtros **u, g, r, i** y **z** de la galaxia **NGC5406**.

Posteriormente:

1. Recorte cada imagen alrededor del centro de la galaxia.
2. Calcule el logaritmo del flujo en cada píxel.
3. Muestre las cinco imágenes en una figura con cinco paneles.
4. Explique por qué la apariencia de la galaxia cambia entre los distintos filtros.

In [ ]:
#Ejercicio 2

import numpy as np
import matplotlib.pyplot as plt
from astroquery.sdss import SDSS
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u

#Filtros fotométricos

bands = ["u", "g", "r", "i", "z"]

#Centro de la galaxia

position = SkyCoord(
    ra=xid["ra"][0] * u.deg,
    dec=xid["dec"][0] * u.deg,
    frame="icrs"
)

#Tamaño del recorte

size = (150, 150)

#Crear figura

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for ax, band in zip(axes, bands):

    #Descargar imagen FITS

    image = SDSS.get_images(matches=xid, band=band)[0]

    data = image[0].data
    wcs = image[0].header

    #Recortar alrededor de la galaxia

    cutout = Cutout2D(
        data,
        position,
        size=size,
        wcs=wcs
    )

    #Logaritmo del flujo

    flux = np.log10(cutout.data - np.min(cutout.data) + 1)

    #Mostrar imagen

    ax.imshow(
        flux,
        origin="lower",
        cmap="gray"
    )

    ax.set_title(f"Filtro {band}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("NGC5406 observada en los filtros u, g, r, i y z")
plt.tight_layout()

plt.show()

Cabe aquí la pregunta de ¿por qué las imágenes son diferentes?, bien, pues cada filtro del SDSS observa un intervalo distinto del espectro electromagnético. Los filtros u y g son más sensibles a la emisión de estrellas jóvenes y calientes, mientras que los filtros r, i y z registran principalmente la emisión de estrellas más frías y viejas. Además, la absorción por polvo interestelar y las diferentes distribuciones espectrales de la población estelar producen variaciones en el brillo y la morfología aparente de la galaxia entre los distintos filtros.

# Ejercicio 3

## Perfil radial de flujo de NGC5406

Calcule el perfil radial de flujo de la galaxia **NGC5406**.

Para ello:

1. Utilice una de las imágenes FITS descargadas anteriormente.
2. Trace diez líneas radiales que partan desde el centro de la galaxia.
3. Obtenga el flujo como función del radio para cada una de estas líneas.
4. Represente todos los perfiles en una misma gráfica.
5. Discuta qué tipo de función describe mejor el comportamiento obtenido.

In [ ]:
#Ejercicio 3

import numpy as np
import matplotlib.pyplot as plt
from astroquery.sdss import SDSS
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u

#Descargar imagen del filtro r

image = SDSS.get_images(matches=xid, band="r")[0]

data = image[0].data

position = SkyCoord(
    ra=xid["ra"][0] * u.deg,
    dec=xid["dec"][0] * u.deg,
    frame="icrs"
)

cutout = Cutout2D(
    data,
    position,
    size=(150,150),
    wcs=image[0].header
)

img = cutout.data

#Centro de la imagen

cy, cx = np.array(img.shape)//2

#Radio máximo permitido

rmax = min(cx, cy)

#Ángulos de las diez líneas

angles = np.linspace(0, 2*np.pi, 10, endpoint=False)

plt.figure(figsize=(8,6))

for theta in angles:

    r = np.arange(rmax)

    x = np.round(cx + r*np.cos(theta)).astype(int)
    y = np.round(cy + r*np.sin(theta)).astype(int)

    mask = (
        (x >= 0) &
        (x < img.shape[1]) &
        (y >= 0) &
        (y < img.shape[0])
    )

    x = x[mask]
    y = y[mask]
    r = r[mask]

    flux = img[y, x]

    plt.plot(r, flux)

plt.xlabel("Radio (pixeles)")
plt.ylabel("Flujo")
plt.title("Perfiles radiales de flujo de NGC5406")
plt.grid(True)

plt.show()

El flujo disminuye conforme aumenta la distancia al centro de la galaxia. Aunque existen pequeñas diferencias entre los diez perfiles debido a la estructura espiral, regiones de formación estelar y ruido observacional, todos presentan una tendencia decreciente similar. Este comportamiento suele describirse mediante un perfil exponencial, característico de galaxias espirales, $$I(r) = I_0 e^{-r/h}$$ donde $I_0$ es el brillo central y $h$ es la longitud de escala del disco. En galaxias elípticas o bulbos dominantes, un perfil de Sérsic proporciona generalmente un ajuste más adecuado.

# Ejercicio 4

## Análisis de la galaxia SDSS J013755.71+010004.9

Repita el procedimiento realizado en los **Ejercicios 2 y 3** para la galaxia **SDSS J013755.71+010004.9**.

En particular:

1. Descargue las imágenes FITS correspondientes a los filtros **u, g, r, i** y **z**.
2. Muestre el logaritmo del flujo en cada uno de los cinco filtros.
3. Calcule el perfil radial de flujo utilizando diez líneas que partan desde el centro de la galaxia.
4. Compare el aspecto de esta galaxia con el de **NGC5406** y explique por qué presentan diferencias morfológicas.

In [ ]:
#Ejercicio 4

import numpy as np
import matplotlib.pyplot as plt
from astroquery.sdss import SDSS
from astropy.coordinates import SkyCoord
from astropy.nddata import Cutout2D
import astropy.units as u

#Nombre de la galaxia

galaxy_name = "SDSS J013755.71+010004.9"

#Consultar objeto

coord = SkyCoord.from_name(galaxy_name)

xid = SDSS.query_region(
    coord,
    spectro=True
)

#Filtros fotométricos

bands = ["u", "g", "r", "i", "z"]

#Mostrar imágenes

fig, axes = plt.subplots(1, 5, figsize=(18,4))

for ax, band in zip(axes, bands):

    image = SDSS.get_images(matches=xid, band=band)[0]

    data = image[0].data

    cutout = Cutout2D(
        data,
        coord,
        size=(150,150),
        wcs=image[0].header
    )

    img = np.log10(cutout.data - np.min(cutout.data) + 1)

    ax.imshow(
        img,
        origin="lower",
        cmap="gray"
    )

    ax.set_title(f"Filtro {band}")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(galaxy_name)
plt.tight_layout()
plt.show()

#Perfil radial usando el filtro r

image = SDSS.get_images(matches=xid, band="r")[0]

cutout = Cutout2D(
    image[0].data,
    coord,
    size=(150,150),
    wcs=image[0].header
)

img = cutout.data

cy, cx = np.array(img.shape)//2

rmax = min(cx, cy)

angles = np.linspace(0, 2*np.pi, 10, endpoint=False)

plt.figure(figsize=(8,6))

for theta in angles:

    r = np.arange(rmax)

    x = np.round(cx + r*np.cos(theta)).astype(int)
    y = np.round(cy + r*np.sin(theta)).astype(int)

    mask = (
        (x >= 0) &
        (x < img.shape[1]) &
        (y >= 0) &
        (y < img.shape[0])
    )

    x = x[mask]
    y = y[mask]
    r = r[mask]

    flux = img[y, x]

    plt.plot(r, flux)

plt.xlabel("Radio (pixeles)")
plt.ylabel("Flujo")
plt.title(f"Perfil radial de {galaxy_name}")
plt.grid(True)

plt.show()

La galaxia SDSS J013755.71+010004.9 presenta una morfología diferente a la de NGC5406, lo que se refleja tanto en las imágenes obtenidas en los distintos filtros como en sus perfiles radiales de flujo. Estas diferencias pueden deberse al tipo morfológico de la galaxia (espiral, elíptica o irregular), a la distribución de sus poblaciones estelares, a la presencia de regiones de formación estelar, al contenido de polvo interestelar y a la orientación con respecto a la línea de visión. En consecuencia, tanto la distribución espacial del brillo como la forma del perfil radial pueden diferir significativamente entre ambas galaxias.